<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 5: Pandas, agrupaciones y combinación</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

## Conceptos

1. Resumir una tabla por grupo con `groupby` y pedir varias métricas a la vez con `agg`.
2. Elegir entre promedio, mediana, suma y recuento según lo que se quiera afirmar, y no por costumbre.
3. Combinar tablas con `merge`, controlando cuántas filas entran y cuántas salen.
4. Detectar lo que un cruce mal hecho hace desaparecer en silencio, y lo que duplica sin avisar.

<h2 style='font-weight:bold; color:#003366;'>Dinámica de clase</h2>

Segunda parte de la misma investigación. Las mesas son las mismas y los casos son los mismos pero cada caso viene con tablas adicionales. 

La puesta en común es contar los hallazgos de la parte 1 y **defender una recomendación** frente al otro grupo.

<h2 style='font-weight:bold; color:#003366;'>1. Teórico: agrupar y cruzar</h2>

Usa los mismos datos que el teórico anterior de `ventas_ecommerce.csv`, para ver las tres funcionalidades nuevas antes de aplicarlas: agrupar, pedir varias métricas juntas y cruzar con otra tabla.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda valor: f"{valor:,.2f}")


def ruta_datos(nombre_archivo, subcarpeta=""):
    """Busca el archivo en las ubicaciones habituales del curso, incluida Colab."""
    candidatas = [
        Path("../Datos") / subcarpeta / nombre_archivo,
        Path("Datos") / subcarpeta / nombre_archivo,
        Path("/content/Datos") / subcarpeta / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")


ventas = pd.read_csv(ruta_datos("ventas_ecommerce.csv"), parse_dates=["fecha"])
ventas["facturacion"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento"])
print("Filas:", len(ventas))

Filas: 8637


### Agrupar

`groupby` parte la tabla en grupos según los valores de una o más columnas y permite aplicar funciones sobre cada grupo.

Cuando usamos **`agg`**, las métricas **resumen cada grupo**, por lo que normalmente obtenemos una fila por grupo. Es útil cuando queremos construir una tabla resumen.

Cuando usamos **`transform`**, la métrica se calcula a nivel de grupo pero **se conserva la cantidad de filas original**. Es útil cuando queremos agregar a cada registro información calculada a partir de su grupo, por ejemplo, el promedio de ventas de su categoría.

Lo que devuelve `groupby` con una agregación tiene la columna de agrupación como índice, no como columna. Eso importa cuando después se quiere cruzar con otra tabla: con `reset_index()` vuelve a ser una columna común.

```python
df.groupby("categoria")["ventas"].agg("mean")
```

```python
df["promedio_categoria"] = df.groupby("categoria")["ventas"].transform("mean")
```

**En resumen:** `agg` **reduce** los datos para obtener un resumen por grupo, mientras que `transform` **mantiene** todas las filas y devuelve el resultado alineado con el DataFrame original.

In [ ]:
# Una métrica por grupo
print(ventas.groupby("categoria")["facturacion"].sum().sort_values(ascending=False)) #sort_values al final para ordenar de mayor a menor

categoria
Electronica    80,541,887.58
Hogar          20,857,983.16
Deportes       14,428,583.41
Indumentaria   12,258,361.23
Libros          2,158,801.75
Name: facturacion, dtype: float64


In [ ]:
# Tabla agrupada con varias métricas de una vez, con nombres elegidos por quien escribe
resumen = ventas.groupby("categoria").agg(
    ordenes=("id_orden", "count"),
    facturacion=("facturacion", "sum"),
    ticket_promedio=("facturacion", "mean"),
    ticket_mediano=("facturacion", "median"),
)

resumen["participacion"] = resumen["facturacion"] / resumen["facturacion"].sum()

# Visualización incorrecta de resultados (por defecto la que hace la IA)
print("\n", resumen.sort_values("facturacion", ascending=False).round(2))


               ordenes   facturacion  ticket_promedio  ticket_mediano  \
categoria                                                              
Electronica      1952 80,541,887.58        41,261.21       38,701.92   
Hogar            1783 20,857,983.16        11,698.25       11,209.40   
Deportes         1516 14,428,583.41         9,517.54        9,082.91   
Indumentaria     2326 12,258,361.23         5,270.15        5,049.09   
Libros           1060  2,158,801.75         2,036.61        1,964.12   

              participacion  
categoria                    
Electronica            0.62  
Hogar                  0.16  
Deportes               0.11  
Indumentaria           0.09  
Libros                 0.02  


In [ ]:
# Visualización correcta
display(resumen.sort_values("facturacion", ascending=False).round(2))

,ordenes,facturacion,ticket_promedio,ticket_mediano,participacion
categoria,,,,,
Electronica,1952,"80,541,887.58","41,261.21","38,701.92",0.62
Hogar,1783,"20,857,983.16","11,698.25","11,209.40",0.16
Deportes,1516,"14,428,583.41","9,517.54","9,082.91",0.11
Indumentaria,2326,"12,258,361.23","5,270.15","5,049.09",0.09
Libros,1060,"2,158,801.75","2,036.61","1,964.12",0.02


**Importante todos las visualizaciones de tablas se deben realizar con display no con print**

Poner el promedio donde correspondía la mediana es el error más común y el más dificil de detectar de los informes de la clase, ya que si no soy conciente que tengo outliers que suben el promedio puedo sobre estimar mi orden normal y sacar proyecciones incorrectas.

In [ ]:
# Crear venta unitaria por categoría

ventas["venta_x_categoria"] = (ventas.groupby("categoria")["id_orden"]
      .transform("count"))

ventas.head()

,id_orden,fecha,categoria,producto,region,canal,unidades,precio_unitario,descuento,costo_envio,cliente_nuevo,facturacion,venta_x_categoria
0,ORD-0000-00000,2024-01-01,Hogar,Silla ergonomica,Montevideo,Marketplace,1,"5,634.83",0.10,224.99,False,"5,071.35",1783
1,ORD-0000-00001,2024-01-01,Indumentaria,Zapatillas,Colonia,Web,3,"2,315.23",0.00,173.71,True,"6,945.69",2326
2,ORD-0000-00002,2024-01-01,Indumentaria,Pantalon,Montevideo,Marketplace,2,"2,495.15",0.10,233.84,False,"4,491.27",2326
3,ORD-0000-00003,2024-01-01,Deportes,Colchoneta,Interior resto,Telefono,1,"4,565.54",0.25,461.22,False,"3,424.15",1516
4,ORD-0000-00004,2024-01-01,Electronica,Notebook,Interior resto,App movil,3,"15,646.05",0.00,322.34,False,"46,938.15",1952


### Unir fuentes de datos

**`merge`** combina tablas relacionando sus filas a partir de una o más columnas en común. Es similar a realizar un **JOIN en SQL** y permite definir cómo conservar las filas:
- `inner` solamente las que coincidan en ambas tablas (intersección de conjuntos). 
- `left` conserva todas las de la tabla de la izquierda y pone nulos en las columnas de la derecha (conjunto izquierdo + intersección).
- `right` igual que lo anterior pero al revés (conjunto derecho + intersección).
- `outer` conserva todos los registros de ambas tablas rellenando con nulos donde no hay cruces (unión de conjuntos).

**`concat`** une tablas **por filas o por columnas**, manteniendo su estructura. Es útil para apilar datasets con las mismas columnas o para agregar nuevas columnas de distintas fuentes.


In [ ]:
margenes_objetivo = pd.DataFrame({
    "categoria": ["Electronica", "Hogar", "Indumentaria", "Libros", "Deportes"],
    "margen_objetivo": [0.18, 0.32, 0.45, 0.28, 0.38],
})

ventas_con_objetivo = ventas.merge(margenes_objetivo, on="categoria", how="left")

print("Filas antes del cruce:", len(ventas))
print("Filas después del cruce:", len(ventas_con_objetivo))
print("Filas sin objetivo asignado:", ventas_con_objetivo["margen_objetivo"].isna().sum())

Filas antes del cruce: 8637
Filas después del cruce: 8637
Filas sin objetivo asignado: 0


El cruce no agregó filas porque la tabla de la derecha tiene un solo registro por categoría. Si tuviera
dos filas para la misma categoría, cada fila de ventas se duplicaría, y la facturación total del informe
saldría inflada sin que nada avise.

De ahí las dos verificaciones que corresponde hacer siempre, y que cuestan dos líneas:

- comparar la cantidad de filas antes y después,
- contar los faltantes en una columna que vino de la tabla derecha, porque cada faltante es una clave que
  no encontró pareja.

Sobre `how`: con `left` se conservan todas las filas de la izquierda y las que no encuentran pareja
quedan con faltantes. Con `inner` se conservan solamente las que encontraron pareja, y las otras
desaparecen sin ningún mensaje. `inner` es el que produce los errores más difíciles de encontrar, porque
el resultado se ve perfecto.

In [ ]:
# Qué pasa cuando la tabla de la derecha tiene claves repetidas
margenes_duplicados = pd.concat([margenes_objetivo, margenes_objetivo.head(2)], ignore_index=True)
display(margenes_duplicados)

# Realizo el mismo cruce que el anterior, pero ahora con claves repetidas en la tabla de la derecha
cruce_inflado = ventas.merge(margenes_duplicados, on="categoria", how="left")

print("Filas después del cruce con claves repetidas:", len(cruce_inflado))
print(f"Facturación real: $U {ventas['facturacion'].sum():,.0f}")
print(f"Facturación inflada: $U {cruce_inflado['facturacion'].sum():,.0f}")

,categoria,margen_objetivo
0,Electronica,0.18
1,Hogar,0.32
2,Indumentaria,0.45
3,Libros,0.28
4,Deportes,0.38
5,Electronica,0.18
6,Hogar,0.32


Filas después del cruce con claves repetidas: 12372
Facturación real: $U 130,245,617
Facturación inflada: $U 231,645,488


Casi el doble de facturación, con un cruce que no dio ningún error. Ese es el motivo por el que las dos
líneas de verificación no son opcionales.

<h2 style='font-weight:bold; color:#003366; text-align:center;'>Para seguir en casa</h2>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 1 *</h3>

Agrupar por dos columnas a la vez y leer el resultado.

Caso A: rentabilidad neta promedio y cantidad de clientes por segmento y tipo de sucursal.
Caso B: margen sobre facturación y cantidad de facturas por línea de producto y canal de venta.

La pregunta a contestar: aparece alguna combinación que sea claramente peor que las demás, y esa
combinación tiene suficientes casos atrás como para afirmarlo.

In [ ]:
# Tu código acá

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
print(clientes.groupby(["segmento", "tipo_sucursal"]).agg(
    clientes=("id_cliente", "count"),
    rentabilidad_promedio=("rentabilidad_neta", "mean"),
).round(0))

# Caso B
resumen = facturas.groupby(["linea_producto", "canal_venta"]).agg(
    facturas=("id_factura", "count"),
    facturacion=("facturacion_real", "sum"),
    margen=("margen_neto", "sum"),
)
resumen["margen_pct"] = resumen["margen"] / resumen["facturacion"]
print(resumen.round(3))
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 2 **</h3>

La evolución mensual, que es la que contesta si el problema está mejorando o empeorando.

Caso A: no hay serie de tiempo de rentabilidad, así que la pregunta es otra. Agrupar las operaciones por
mes y canal y ver si la mezcla de canales cambia dentro del mes observado.

Caso B: agrupar la facturación y el margen por mes, pegarle la tabla de costos fijos y calcular el
resultado operativo mensual, que es el margen neto menos el costo fijo del mes. Después contestar la
pregunta del dueño con evidencia, comparando los primeros seis meses observados contra los últimos seis:
la facturación, el margen neto y el resultado operativo se mueven en la misma dirección y en la misma
magnitud.

In [ ]:
# Tu código acá

<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b> (abrila solo después de intentarlo)</summary>

Caso B: la clave del cruce es la columna de mes, que en `facturas` ya se creó como texto con formato
año-mes y en `costos_fijos` tiene el mismo formato. Conviene verificar que los dos lados tengan la misma
cantidad de valores distintos antes de cruzar, porque un mes sin facturación no va a aparecer en el
`groupby` y va a quedar sin costo fijo asignado, lo que da un resultado operativo equivocado por
omisión.

Para ver si algo crece, ordenar por mes y mirar la serie completa suele alcanzar. Comparar solamente el
primer mes contra el último es lo que hace que dos personas mirando los mismos datos lleguen a
conclusiones opuestas.

</details>

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
operaciones["semana"] = operaciones["fecha"].dt.isocalendar().week
print(pd.crosstab(operaciones["semana"], operaciones["canal"], normalize="index").round(3))

# Caso B
mensual = facturas.groupby("mes").agg(
    facturacion=("facturacion_real", "sum"),
    margen_neto=("margen_neto_real", "sum"),
    facturas=("id_factura", "count"),
).reset_index()

mensual = mensual.merge(costos_fijos, on="mes", how="left")
print("Meses sin costo fijo asignado:", mensual["costo_fijo_uyu"].isna().sum())

mensual["resultado_operativo"] = mensual["margen_neto"] - mensual["costo_fijo_uyu"]
mensual["margen_pct"] = mensual["margen_neto"] / mensual["facturacion"]
print(mensual[["mes", "facturacion", "margen_neto", "costo_fijo_uyu",
               "resultado_operativo", "margen_pct"]].round(2).to_string(index=False))
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 3 ***</h3>

Reconstruir el resultado principal de la mesa **sin usar la tabla que se agregó hoy**, y comparar las dos
versiones en una sola tabla de dos filas.

Caso A: la rentabilidad por segmento sin el costo de atención, contra la rentabilidad con el costo.
Caso B: el margen por sector con el tipo de cambio fijo en 41, contra el margen con la cotización mensual.

La entrega es la tabla comparativa y una frase sobre qué decisión habría tomado el que pidió el análisis
si le hubiéramos entregado la primera versión.

In [ ]:
# Tu código acá

<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b> (abrila solo después de intentarlo)</summary>

No hace falta nada nuevo: son dos `groupby` sobre las mismas filas usando columnas distintas, y después
juntar los dos resultados. Para ponerlos lado a lado, la opción más simple es armar un DataFrame nuevo
con los dos resultados como columnas, aprovechando que los dos tienen el mismo índice de grupos.

El sentido del ejercicio es el que va a aparecer en el proyecto final: casi todos los errores graves de
análisis no son errores de código, son análisis correctos hechos sobre una tabla incompleta.

</details>

<details>
<summary><b>Ver solución</b></summary>

```python
# Caso A
comparacion = pd.DataFrame({
    "sin_costo_atencion": clientes.groupby("segmento")["ingreso_anual"].sum(),
    "con_costo_atencion": clientes.groupby("segmento")["rentabilidad_neta"].sum(),
})
comparacion["diferencia_pct"] = (comparacion["con_costo_atencion"]
                                 / comparacion["sin_costo_atencion"] - 1)
print(comparacion.round(3))

# Caso B
facturas["margen_neto_fijo"] = (facturas["facturacion"]
                                - facturas["unidades"] * facturas["costo_unitario"]
                                * facturas["factor_moneda"]
                                - facturas["costo_flete"])
comparacion = pd.DataFrame({
    "tipo_cambio_fijo": facturas.groupby("sector")["margen_neto_fijo"].sum(),
    "cotizacion_mensual": facturas.groupby("sector")["margen_neto_real"].sum(),
})
comparacion["diferencia_pct"] = (comparacion["cotizacion_mensual"]
                                 / comparacion["tipo_cambio_fijo"] - 1)
print(comparacion.round(3))
```

</details>

<h2 style='font-weight:bold; color:#003366;'>Anexo: Conceptos base</h2>

| Qué se quiere | Cómo se escribe |
|---|---|
| Una métrica por grupo | `tabla.groupby("grupo")["monto"].sum()` |
| Varias métricas por grupo | `tabla.groupby("grupo").agg(n=("id", "count"), total=("monto", "sum"))` |
| Agrupar por dos columnas | `tabla.groupby(["grupo", "subgrupo"])["monto"].mean()` |
| Volver el índice a columna | `resultado.reset_index()` |
| Columna calculada sobre el resumen | `resumen["pct"] = resumen["a"] / resumen["a"].sum()` |
| Ordenar el resumen | `resumen.sort_values("total", ascending=False)` |
| Pegar otra tabla | `izquierda.merge(derecha, on="clave", how="left")` |
| Pegar por claves con nombres distintos | `izquierda.merge(derecha, left_on="a", right_on="b", how="left")` |
| Verificar el cruce | comparar `len()` antes y después, y `isna().sum()` de una columna de la derecha |
| Filas de la izquierda sin pareja | `izquierda[~izquierda["clave"].isin(derecha["clave"])]` |
| Composición porcentual cruzada | `pd.crosstab(tabla["a"], tabla["b"], normalize="index")` |
| Valores distintos por grupo | `tabla.groupby("grupo")["cliente"].nunique()` |
| Suma acumulada | `serie.cumsum() / serie.sum()` |
| Apilar dos tablas con las mismas columnas | `pd.concat([tabla1, tabla2], ignore_index=True)` |